# Plot saved training and test results

Set **RUN_DIR** to one single-training or K-Fold run folder, then run the cells. No training or inference is performed. Use the **rnet** kernel.

Losses are shown separately for every saved fold, using completed epochs (epoch numbering starts at 1). Boxplots use **postprocessed full-volume** metrics: fixed-split `test` and CV outer `holdout` by default. Native patch metrics are pooled cohort measurements and cannot provide subject boxplots.

Each dot is one animal per condition, modality, region and evaluation role. Repeated predictions of a scan are averaged first; longitudinal scans are then averaged within that animal and modality. Cohorts, roles and units stay separate. Missing annotations and undefined scores are omitted, while genuine zero scores and finite both-empty conventions are retained.

Plots display inline without widgets or HTML. PNG/PDF/SVG exports use Matplotlib and do not require Kaleido.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from datetime import datetime
import hashlib
import json
import math

from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns

from src.evaluation.run_plots import animal_metric_points, load_run_plot_data

sns.set_theme(style="whitegrid", context="notebook")

## 1. Choose a run and plotting settings

Use the run containing `experiment.json` and `fold_0`, rather than a model-group folder. Loss history is read from committed recovery state when available, so interrupted runs are supported.

`METRICS_FILE=None` selects one existing aggregate report from the run, falling back to per-fold evaluation files. To plot a stopped-checkpoint evaluation, set its exact `full_volume_metrics.csv` path explicitly; no latest evaluation is selected. Keep `RUN_DIR` pointing to the associated training run for its loss histories and scan metadata.

Leave filters as `None` to keep all available values. Choose region names from the list printed below. `LOSS_KEYS=None` includes total loss and every saved head loss. Older rat notebook 04 runs store the held-out cohort as `validation`: explicitly select that role for historical runs. New fixed-epoch notebook 04 runs use `holdout`, included by default. This notebook never relabels saved roles.

In [ ]:
RUN_DIR: Path | None = None  # Example: REPO_ROOT / "results/d1_3drnet_kfold/RUN_..."
METRICS_FILE: Path | None = None  # Optional exact stopped-checkpoint full_volume_metrics.csv.

METRIC: str = "Dice"  # Examples: Dice, HD95, ASSD, Sensitivity, Precision, AVE.
ROLES: tuple[str, ...] = ("test", "holdout")
FOLDS: tuple[int, ...] | None = None
REGIONS: tuple[str, ...] | None = None
DATASETS: tuple[str, ...] | None = None
CONDITIONS: tuple[str, ...] | None = None
MODALITIES: tuple[str, ...] | None = None

LOSS_KEYS: tuple[str, ...] | None = ("loss", "val_loss")
LOSS_LOG_SCALE: bool = False
SHOW_SUBJECT_POINTS: bool = True
METRIC_LIMITS: tuple[float, float] | None = None  # For example (0.0, 1.0) for Dice.
FIGURE_DPI: int = 300

EXPORT_RESULTS: bool = False
OUTPUT_DIR: Path | None = None  # None uses RUN_DIR / "plots"; each export creates a new snapshot.
EXPORT_FORMATS: tuple[str, ...] = ("png", "pdf", "svg")

## 2. Load saved data and inspect coverage

The table below counts raw saved measurements by annotation/empty-mask status. Inspect coverage before comparing regions: partial annotations can give different animal counts. Missing condition or modality remains `unknown`; known values are joined from saved manifests/contracts without guessing from filenames.

In [ ]:
saved = None
history = pd.DataFrame()
selected_metrics = pd.DataFrame()
animal_points = pd.DataFrame()
metric_summary = pd.DataFrame()
coverage = pd.DataFrame()
figures: dict[str, Figure] = {}

if RUN_DIR is None:
    print("Set RUN_DIR, then rerun this cell and the plotting cells.")
else:
    saved = load_run_plot_data(run_dir=RUN_DIR, metrics_file=METRICS_FILE)
    history = saved.history.copy()
    selected_metrics = saved.metrics.copy()

    if FOLDS is not None and not history.empty:
        history = history.loc[history["Fold"].isin(FOLDS)].copy()

    if not history.empty:
        print("Available loss keys:", sorted(history["Loss"].unique()))
        epoch_counts = history.groupby("Fold")["Epoch"].max().rename("CompletedEpochs")
        display(epoch_counts.to_frame())
    else:
        print("No loss history found for the selected folds.")

    if not selected_metrics.empty:
        print("Available metrics:", sorted(selected_metrics["Metric"].unique()))
        print("Available roles:", sorted(selected_metrics["DatasetRole"].unique()))
        print("Available regions:", sorted(selected_metrics["Region"].unique()))

        filters = {
            "Fold": FOLDS, "Region": REGIONS, "Dataset": DATASETS,
            "Condition": CONDITIONS, "Modality": MODALITIES,
        }
        for column, values in filters.items():
            if values is not None:
                selected_metrics = selected_metrics.loc[selected_metrics[column].isin(values)].copy()

        selection = selected_metrics["Metric"].eq(METRIC) & selected_metrics["DatasetRole"].isin(ROLES)
        selected_metrics = selected_metrics.loc[selection].copy()
        coverage_keys = ["Dataset", "DatasetRole", "Condition", "Modality", "Region", "Status"]
        coverage = selected_metrics.groupby(coverage_keys, dropna=False).agg(
            Animals=("Subject", "nunique"), Scans=("Scan", "nunique"), Rows=("Value", "size"),
        ).reset_index()
        display(coverage)
    else:
        print("No full-volume report found. Loss plots remain available; evaluate saved weights or set METRICS_FILE.")

## 3. Training and validation loss curves

Every panel represents one fold. Curves retain their actual completed duration; shorter folds are not padded or extrapolated. Change `LOSS_KEYS` to inspect ROI, brain or classifier losses, whose scales may differ. No smoothing is applied.

In [ ]:
if not history.empty:
    loss_history = history.copy()
    if LOSS_KEYS is not None:
        loss_history = loss_history.loc[loss_history["Loss"].isin(LOSS_KEYS)].copy()
    if loss_history.empty:
        raise ValueError("No saved loss matches LOSS_KEYS; inspect the available keys above")

    fold_ids = sorted(loss_history["Fold"].unique())
    columns = min(2, len(fold_ids))
    rows = math.ceil(len(fold_ids) / columns)
    figure_size = (7 * columns, 4 * rows)
    loss_figure, axes = plt.subplots(nrows=rows, ncols=columns, figsize=figure_size, squeeze=False)

    for axis, fold in zip(axes.flat, fold_ids):
        fold_history = loss_history.loc[loss_history["Fold"].eq(fold)]
        for name, curve in fold_history.groupby("Loss", sort=False):
            curve = curve.sort_values("Epoch")
            line_style = "-"
            if name.startswith("val_"):
                line_style = "--"
            axis.plot(curve["Epoch"], curve["Value"], label=name, linestyle=line_style, linewidth=1.5)

        axis.set(title=f"Fold {fold}", xlabel="Epoch", ylabel="Loss")
        if LOSS_LOG_SCALE:
            axis.set_yscale("log")
        axis.legend(fontsize=9)

    for axis in list(axes.flat)[len(fold_ids):]:
        axis.set_visible(False)

    loss_figure.suptitle("Saved training and validation losses")
    loss_figure.tight_layout()
    figures["loss_curves"] = loss_figure
    display(loss_figure)
    plt.close(loss_figure)

## 4. Chosen test metric by condition and modality

Regions are on the x-axis, condition sets the color, and modalities have separate columns. Rows separate dataset, evaluation role and unit. Dots and boxplots use the same animal averages. An animal appearing in several modalities contributes once to each modality, rather than being treated as several independent animals in a pooled score.

The accompanying summary is descriptive: mean, SD, median and range across the plotted animals. It is not a confidence interval or an average of fold averages. `Folds` in the points table records how many models contributed to an animal's selected measurements.

In [ ]:
if saved is not None and not saved.metrics.empty:
    animal_points = animal_metric_points(metrics=selected_metrics, metric=METRIC, roles=ROLES)
    summary_keys = ["Dataset", "DatasetRole", "Condition", "Modality", "Head", "Region", "Metric", "Unit"]
    metric_summary = animal_points.groupby(summary_keys, dropna=False).agg(
        Animals=("Subject", "nunique"), Scans=("Scans", "sum"),
        Mean=("Value", "mean"), SD=("Value", "std"), Median=("Value", "median"),
        Minimum=("Value", "min"), Maximum=("Value", "max"),
    ).reset_index()
    display(metric_summary)

    plot_data = animal_points.copy()
    plot_data["Panel"] = plot_data["Dataset"] + " / " + plot_data["DatasetRole"] + " / " + plot_data["Unit"]
    region_order = list(plot_data["Region"].unique())
    if REGIONS is not None:
        region_order = list(REGIONS)
    condition_order = sorted(plot_data["Condition"].unique())
    modality_order = sorted(plot_data["Modality"].unique())
    panel_order = list(plot_data["Panel"].unique())
    palette_values = sns.color_palette("colorblind", n_colors=len(condition_order))
    palette = dict(zip(condition_order, palette_values))
    facet_width = max(5.0, len(region_order) * 0.8)
    facet_aspect = facet_width / 4.5
    show_outliers = not SHOW_SUBJECT_POINTS

    grid = sns.catplot(
        data=plot_data, x="Region", y="Value", hue="Condition", kind="box",
        row="Panel", col="Modality", order=region_order, hue_order=condition_order,
        row_order=panel_order, col_order=modality_order, palette=palette,
        height=4.5, aspect=facet_aspect, sharex=False, sharey="row",
        showfliers=show_outliers, legend=False,
    )
    if SHOW_SUBJECT_POINTS:
        grid.map_dataframe(
            sns.stripplot, x="Region", y="Value", hue="Condition", order=region_order,
            hue_order=condition_order, palette=palette, dodge=True, jitter=0.12,
            size=3.5, alpha=0.65, legend=False,
        )

    grid.set_axis_labels("Region", METRIC)
    grid.set_titles(row_template="{row_name}", col_template="{col_name}")
    for axis in grid.axes.flat:
        axis.tick_params(axis="x", labelrotation=45)
        if METRIC_LIMITS is not None:
            axis.set_ylim(METRIC_LIMITS)
    legend_handles = [Patch(facecolor=palette[name], label=name) for name in condition_order]
    grid.figure.legend(
        handles=legend_handles, title="Condition", loc="upper center",
        bbox_to_anchor=(0.5, 1.0), ncol=len(condition_order), frameon=False,
    )
    grid.figure.suptitle(f"{METRIC}: full-volume animal means", y=1.05)
    grid.figure.tight_layout(rect=(0.0, 0.0, 1.0, 0.93))
    figures["metric_boxplots"] = grid.figure
    display(grid.figure)
    plt.close(grid.figure)

## 5. Export figures and their exact plotting data

Set `EXPORT_RESULTS=True` after inspecting the plots. A new snapshot contains PNG/PDF/SVG figures, loss observations, selected raw measurements (including missing statuses), animal points, summary/coverage CSVs and a JSON record of settings and input SHA256 hashes. Existing results are not overwritten. Vector PDF/SVG files are suitable for manuscript layout.

In [ ]:
if EXPORT_RESULTS:
    if saved is None or not figures:
        raise ValueError("Select a run and create at least one figure before exporting")
    allowed_formats = {"png", "pdf", "svg"}
    if not EXPORT_FORMATS or not set(EXPORT_FORMATS).issubset(allowed_formats):
        raise ValueError("Choose one or more EXPORT_FORMATS from png, pdf and svg")

    export_root = OUTPUT_DIR
    if export_root is None:
        export_root = RUN_DIR / "plots"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    export_dir = export_root / timestamp
    export_dir.mkdir(parents=True, exist_ok=False)

    for name, figure in figures.items():
        for extension in EXPORT_FORMATS:
            destination = export_dir / f"{name}.{extension}"
            figure.savefig(destination, dpi=FIGURE_DPI, bbox_inches="tight")

    export_tables = {
        "loss_history": history, "selected_scan_metrics": selected_metrics,
        "animal_metric_points": animal_points, "metric_summary": metric_summary,
        "annotation_coverage": coverage,
    }
    for name, table in export_tables.items():
        destination = export_dir / f"{name}.csv"
        table.to_csv(destination, index=False)

    input_records: list[dict[str, str]] = []
    for path in saved.sources:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        input_records.append({"Path": str(path), "SHA256": digest})
    settings = {
        "run_dir": str(RUN_DIR.resolve()), "metrics_file": METRICS_FILE,
        "metric": METRIC, "roles": ROLES, "folds": FOLDS, "regions": REGIONS,
        "datasets": DATASETS, "conditions": CONDITIONS, "modalities": MODALITIES,
        "loss_keys": LOSS_KEYS, "loss_log_scale": LOSS_LOG_SCALE,
        "show_subject_points": SHOW_SUBJECT_POINTS, "metric_limits": METRIC_LIMITS,
        "formats": EXPORT_FORMATS, "dpi": FIGURE_DPI,
        "aggregation": "mean repeated predictions per scan, then mean scans per animal within modality/condition/region/role",
        "sources": input_records,
    }
    settings_file = export_dir / "plot_settings.json"
    settings_file.write_text(json.dumps(settings, indent=2, default=str), encoding="utf-8")
    print("Exported figures and plotting data to:", export_dir)
else:
    print("Set EXPORT_RESULTS=True to export the displayed figures and data.")